# 28 — Bidirectional all-year spatiotemporal model v4

Controlled comparison with V2: identical data, sampling, augmentation, sparse pooling, loss, and validation contract, with bidirectional temporal context as the only modeling change. The official test remains untouched.


In [1]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
from pathlib import Path

PACKAGE_VERSION = "0.1.10"
VALIDATION_FRACTION = 0.20
VALIDATION_SEED = 20260913
TRAINING_SEED = 20260928
YEARS = tuple(range(2013, 2023))
VARIABLES = (
    "DBZ",
    "KDP",
    "RHOHV",
    "VEL",
    "WIDTH",
    "ZDR",
)
CHANNEL_ORDER = tuple(
    f"{variable}_sweep_{sweep}"
    for variable in VARIABLES
    for sweep in range(2)
)
CHANNEL_COUNT = len(CHANNEL_ORDER)

FILE_BATCH_SIZE = 8
NUM_WORKERS = 12
MAX_EPOCHS = 40
EARLY_STOPPING_PATIENCE = 6
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
SCHEDULER_PATIENCE = 2
SCHEDULER_FACTOR = 0.5
MINIMUM_LEARNING_RATE = 1e-5
POOLING_TOPK_FRACTION = 0.05
OVERALL_SCORE_WEIGHT = 0.70

BACKUP_ROOT = Path(
    "/content/drive/MyDrive/TorNet_Backup"
)
PACKAGE_PATH = (
    BACKUP_ROOT
    / "packages"
    / f"tornet_detection-{PACKAGE_VERSION}-py3-none-any.whl"
)
MANIFESTS_ROOT = BACKUP_ROOT / "manifests"
NORMALIZATION_PATH = (
    BACKUP_ROOT
    / "experiments"
    / "all_year_all6_baseline_v1"
    / "normalization.json"
)
EXPERIMENT_DIRECTORY = (
    BACKUP_ROOT
    / "experiments"
    / "spatiotemporal_bidirectional_all_year_v4"
)
LAST_CHECKPOINT_PATH = (
    EXPERIMENT_DIRECTORY
    / "last_checkpoint.pt"
)
BEST_CHECKPOINT_PATH = (
    EXPERIMENT_DIRECTORY
    / "best_model.pt"
)
HISTORY_PATH = (
    EXPERIMENT_DIRECTORY
    / "training_history.csv"
)
VALIDATION_METRICS_PATH = (
    EXPERIMENT_DIRECTORY
    / "validation_metrics.json"
)
EXTRACTION_ROOT = Path(
    "/content/tornet_bidirectional_all_year_v4"
)

for path in (
    PACKAGE_PATH,
    MANIFESTS_ROOT,
    NORMALIZATION_PATH,
):
    if not path.exists():
        raise FileNotFoundError(path)

if VALIDATION_METRICS_PATH.exists():
    raise FileExistsError(
        "Experiment already completed: "
        f"{VALIDATION_METRICS_PATH}"
    )

EXPERIMENT_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True,
)

print("package:", PACKAGE_PATH)
print("normalization:", NORMALIZATION_PATH)
print("experiment:", EXPERIMENT_DIRECTORY)
print(
    "resume checkpoint:",
    LAST_CHECKPOINT_PATH.exists(),
)


package: /content/drive/MyDrive/TorNet_Backup/packages/tornet_detection-0.1.10-py3-none-any.whl
normalization: /content/drive/MyDrive/TorNet_Backup/experiments/all_year_all6_baseline_v1/normalization.json
experiment: /content/drive/MyDrive/TorNet_Backup/experiments/spatiotemporal_bidirectional_all_year_v4
resume checkpoint: True


In [3]:
import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "netCDF4>=1.7",
        "scikit-learn>=1.5",
    ],
    check=True,
)

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--no-deps",
        "--force-reinstall",
        str(PACKAGE_PATH),
    ],
    check=True,
)

CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '--no-deps', '--force-reinstall', '/content/drive/MyDrive/TorNet_Backup/packages/tornet_detection-0.1.10-py3-none-any.whl'], returncode=0)

In [4]:
import datetime
import json
import random
import shutil
import tarfile
import time

import numpy as np
import pandas as pd
import torch
from torch import nn
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from torch.utils.data import (
    DataLoader,
    Dataset,
)

import tornado_detection
from tornado_detection.data import (
    assign_model_splits,
    load_canonical_frame_index,
    read_spatiotemporal_netcdf_file,
)
from tornado_detection.models import (
    SpatiotemporalTornadoDetector,
)

if tornado_detection.__version__ != PACKAGE_VERSION:
    raise RuntimeError(
        "Unexpected package version: "
        f"{tornado_detection.__version__}"
    )

if not torch.cuda.is_available():
    raise RuntimeError(
        "Select an A100 GPU runtime and run all cells"
    )

if not torch.cuda.is_bf16_supported():
    raise RuntimeError(
        "The selected GPU does not support bfloat16"
    )

random.seed(TRAINING_SEED)
np.random.seed(TRAINING_SEED)
torch.manual_seed(TRAINING_SEED)
torch.cuda.manual_seed_all(TRAINING_SEED)

device = torch.device("cuda")

normalization = json.loads(
    NORMALIZATION_PATH.read_text()
)

expected_normalization = {
    "package_version": "0.1.7",
    "years": list(YEARS),
    "model_split": "train",
    "validation_fraction": (
        VALIDATION_FRACTION
    ),
    "validation_seed": VALIDATION_SEED,
    "training_frame_count": 547_672,
    "training_file_count": 136_918,
    "training_positive_frame_count": 18_230,
    "variables": list(VARIABLES),
    "channel_order": list(CHANNEL_ORDER),
    "tensor_shape": [
        120,
        240,
        CHANNEL_COUNT,
    ],
}

mismatches = {
    key: {
        "expected": expected,
        "actual": normalization.get(key),
    }
    for key, expected
    in expected_normalization.items()
    if normalization.get(key) != expected
}

if mismatches:
    raise AssertionError(
        "Normalization provenance mismatch: "
        f"{mismatches}"
    )

means = np.asarray(
    normalization["means"],
    dtype=np.float32,
).reshape(
    len(VARIABLES),
    2,
).T
stds = np.asarray(
    normalization["standard_deviations"],
    dtype=np.float32,
).reshape(
    len(VARIABLES),
    2,
).T

assert means.shape == (2, 6)
assert stds.shape == (2, 6)
assert np.isfinite(means).all()
assert np.isfinite(stds).all()
assert (stds > 0).all()

canonical = load_canonical_frame_index(
    MANIFESTS_ROOT
)
assigned = assign_model_splits(
    canonical,
    validation_fraction=VALIDATION_FRACTION,
    seed=VALIDATION_SEED,
)

train_rows = (
    assigned.loc[
        assigned["model_split"].eq("train")
    ]
    .sort_values(
        [
            "archive_member",
            "frame_index",
        ]
    )
)
validation_rows = (
    assigned.loc[
        assigned["model_split"].eq("validation")
    ]
    .sort_values(
        [
            "archive_member",
            "frame_index",
        ]
    )
)
test_rows = assigned.loc[
    assigned["model_split"].eq("test")
]

assert len(train_rows) == 547_672
assert int(
    train_rows["frame_label"].sum()
) == 18_230
assert len(validation_rows) == 138_892
assert int(
    validation_rows["frame_label"].sum()
) == 4_200
assert len(test_rows) == 125_868
assert int(
    test_rows["frame_label"].sum()
) == 3_909

for rows in (
    train_rows,
    validation_rows,
):
    counts = rows.groupby(
        "archive_member"
    ).size()

    if not counts.eq(4).all():
        raise AssertionError(
            "Every selected file must "
            "contain four frames"
        )


def build_records(rows):
    return [
        (
            member,
            group["frame_label"]
            .astype(np.uint8)
            .to_numpy(),
        )
        for member, group
        in rows.groupby(
            "archive_member",
            sort=True,
        )
    ]


train_records = build_records(
    train_rows
)
validation_records = build_records(
    validation_rows
)
validation_years = validation_rows[
    "year"
].to_numpy(dtype=np.int64)

assert len(train_records) == 136_918
assert len(validation_records) == 34_723

required_by_year = {
    year: set(
        assigned.loc[
            assigned["year"].eq(year)
            & assigned["model_split"].isin(
                [
                    "train",
                    "validation",
                ]
            ),
            "archive_member",
        ].unique()
    )
    for year in YEARS
}

print("GPU:", torch.cuda.get_device_name(0))
print("torch:", torch.__version__)
print(
    "train files/frames:",
    len(train_records),
    len(train_rows),
)
print(
    "validation files/frames:",
    len(validation_records),
    len(validation_rows),
)
print(
    "official test untouched:",
    len(test_rows),
)


GPU: NVIDIA A100-SXM4-40GB
torch: 2.11.0+cu128
train files/frames: 136918 547672
validation files/frames: 34723 138892
official test untouched: 125868


In [5]:
if EXTRACTION_ROOT.exists():
    shutil.rmtree(EXTRACTION_ROOT)

EXTRACTION_ROOT.mkdir(
    parents=True,
    exist_ok=False,
)

staging = []

for year in YEARS:
    drive_archive = (
        BACKUP_ROOT
        / f"tornet_{year}.tar.gz"
    )
    local_archive = Path(
        f"/content/tornet_{year}.tar.gz"
    )

    if not drive_archive.is_file():
        raise FileNotFoundError(
            drive_archive
        )

    if local_archive.exists():
        local_archive.unlink()

    copy_started = (
        time.perf_counter()
    )

    shutil.copyfile(
        drive_archive,
        local_archive,
    )

    copy_seconds = (
        time.perf_counter()
        - copy_started
    )

    required = required_by_year[year]
    extracted = set()
    extract_started = (
        time.perf_counter()
    )

    with tarfile.open(
        local_archive,
        mode="r:gz",
    ) as archive:
        for member in archive:
            if (
                not member.isfile()
                or member.name
                not in required
            ):
                continue

            destination = (
                EXTRACTION_ROOT
                / member.name
            )
            destination.parent.mkdir(
                parents=True,
                exist_ok=True,
            )

            source_file = (
                archive.extractfile(
                    member
                )
            )

            if source_file is None:
                raise RuntimeError(
                    member.name
                )

            with (
                source_file,
                destination.open("wb")
                as output_file,
            ):
                shutil.copyfileobj(
                    source_file,
                    output_file,
                    length=1024 * 1024,
                )

            extracted.add(
                member.name
            )

    extraction_seconds = (
        time.perf_counter()
        - extract_started
    )
    missing = required - extracted

    if missing:
        raise RuntimeError(
            f"Year {year} missing: "
            f"{sorted(missing)[:10]}"
        )

    local_archive.unlink()

    staging.append(
        {
            "year": year,
            "file_count": len(
                extracted
            ),
            "copy_seconds": (
                copy_seconds
            ),
            "extraction_seconds": (
                extraction_seconds
            ),
        }
    )

    print(
        f"year={year} "
        f"staged_files="
        f"{len(extracted):,} "
        f"copy={copy_seconds:.1f}s "
        f"extract="
        f"{extraction_seconds:.1f}s"
    )

print(
    "total staged files:",
    sum(
        row["file_count"]
        for row in staging
    ),
)

staged_bytes = sum(
    path.stat().st_size
    for path
    in EXTRACTION_ROOT.rglob("*.nc")
)

print(
    "staged GiB:",
    round(
        staged_bytes / 1024**3,
        3,
    ),
)

year=2013 staged_files=3,498 copy=33.1s extract=11.0s
year=2014 staged_files=17,446 copy=156.4s extract=51.2s
year=2015 staged_files=19,721 copy=305.9s extract=71.0s
year=2016 staged_files=18,791 copy=173.1s extract=56.2s
year=2017 staged_files=16,927 copy=281.5s extract=51.3s
year=2018 staged_files=15,353 copy=155.4s extract=44.3s
year=2019 staged_files=20,588 copy=180.6s extract=88.8s
year=2020 staged_files=18,147 copy=279.2s extract=70.0s
year=2021 staged_files=19,032 copy=262.7s extract=72.1s
year=2022 staged_files=22,138 copy=251.2s extract=70.7s
total staged files: 171641
staged GiB: 128.017


In [6]:
class SequenceDataset(Dataset):
    def __init__(
        self,
        file_records,
        root,
        channel_means,
        channel_stds,
        *,
        augment_azimuth=False,
    ):
        self.records = file_records
        self.root = root
        self.augment_azimuth = (
            augment_azimuth
        )
        self.means = channel_means.reshape(
            1,
            2,
            6,
            1,
            1,
        )
        self.stds = channel_stds.reshape(
            1,
            2,
            6,
            1,
            1,
        )

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        (
            member,
            expected_labels,
        ) = self.records[index]

        sequence = (
            read_spatiotemporal_netcdf_file(
                self.root / member,
                variables=VARIABLES,
            )
        )

        np.testing.assert_array_equal(
            sequence.labels,
            expected_labels,
        )

        values = sequence.values
        finite_mask = sequence.finite_mask
        range_folded_mask = (
            sequence.range_folded_mask
        )

        if self.augment_azimuth:
            shift = int(
                torch.randint(
                    values.shape[-2],
                    size=(),
                ).item()
            )
            values = np.roll(
                values,
                shift,
                axis=-2,
            )
            finite_mask = np.roll(
                finite_mask,
                shift,
                axis=-2,
            )
            range_folded_mask = np.roll(
                range_folded_mask,
                shift,
                axis=-2,
            )

        normalized = (
            (
                values
                - self.means
            )
            / self.stds
        ).astype(
            np.float32,
            copy=False,
        )

        return (
            torch.from_numpy(normalized),
            torch.from_numpy(
                finite_mask.copy()
            ),
            torch.from_numpy(
                range_folded_mask.copy()
            ),
            torch.from_numpy(
                sequence.coordinates
            ),
            torch.from_numpy(
                sequence.labels.astype(
                    np.float32
                )
            ),
        )


train_dataset = SequenceDataset(
    train_records,
    EXTRACTION_ROOT,
    means,
    stds,
    augment_azimuth=True,
)
validation_dataset = SequenceDataset(
    validation_records,
    EXTRACTION_ROOT,
    means,
    stds,
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=FILE_BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True,
)

model = (
    SpatiotemporalTornadoDetector(
        pooling_topk_fraction=(
            POOLING_TOPK_FRACTION
        ),
        bidirectional_temporal=True,
    )
    .to(device)
)

train_positive_count = int(
    train_rows["frame_label"].sum()
)
train_negative_count = (
    len(train_rows)
    - train_positive_count
)
positive_weight = torch.tensor(
    train_negative_count
    / train_positive_count,
    device=device,
)

criterion = nn.BCEWithLogitsLoss(
    pos_weight=positive_weight
)
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)
scheduler = (
    torch.optim.lr_scheduler
    .ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=SCHEDULER_FACTOR,
        patience=SCHEDULER_PATIENCE,
        min_lr=MINIMUM_LEARNING_RATE,
    )
)

print(
    "parameters:",
    f"{sum(parameter.numel() for parameter in model.parameters()):,}",
)
print(
    "positive weight:",
    float(positive_weight.cpu()),
)
print(
    "batch size:",
    FILE_BATCH_SIZE,
)


parameters: 2,970,049
positive weight: 29.042346954345703
batch size: 8


In [7]:
def move_batch_to_device(batch):
    (
        values,
        finite_mask,
        range_folded_mask,
        coordinates,
        targets,
    ) = batch

    return (
        values.to(
            device,
            non_blocking=True,
        ),
        finite_mask.to(
            device,
            non_blocking=True,
        ),
        range_folded_mask.to(
            device,
            non_blocking=True,
        ),
        coordinates.to(
            device,
            non_blocking=True,
        ),
        targets.to(
            device,
            non_blocking=True,
        ),
    )


def evaluate(loader):
    model.eval()

    labels = []
    probabilities = []
    loss_sum = 0.0
    count = 0

    with torch.no_grad():
        for batch in loader:
            (
                values,
                finite_mask,
                range_folded_mask,
                coordinates,
                targets,
            ) = move_batch_to_device(
                batch
            )

            with torch.autocast(
                device_type="cuda",
                dtype=torch.bfloat16,
            ):
                logits = model(
                    values,
                    finite_mask,
                    range_folded_mask,
                    coordinates,
                )
                loss = criterion(
                    logits,
                    targets,
                )

            size = int(
                targets.numel()
            )
            loss_sum += (
                float(loss.cpu())
                * size
            )
            count += size

            labels.extend(
                targets.cpu()
                .numpy()
                .reshape(-1)
                .tolist()
            )
            probabilities.extend(
                logits.sigmoid()
                .float()
                .cpu()
                .numpy()
                .reshape(-1)
                .tolist()
            )

    labels = np.asarray(
        labels,
        dtype=np.int64,
    )
    probabilities = np.asarray(
        probabilities,
        dtype=np.float64,
    )

    return (
        loss_sum / count,
        float(
            average_precision_score(
                labels,
                probabilities,
            )
        ),
        float(
            roc_auc_score(
                labels,
                probabilities,
            )
        ),
        labels,
        probabilities,
    )


def validation_scores(
    labels,
    probabilities,
):
    per_year = {}

    for year in YEARS:
        selected = (
            validation_years == year
        )
        per_year[year] = float(
            average_precision_score(
                labels[selected],
                probabilities[selected],
            )
        )

    macro = float(
        np.mean(
            list(per_year.values())
        )
    )
    overall = float(
        average_precision_score(
            labels,
            probabilities,
        )
    )
    selection = (
        OVERALL_SCORE_WEIGHT * overall
        + (
            1.0
            - OVERALL_SCORE_WEIGHT
        )
        * macro
    )

    return (
        overall,
        macro,
        selection,
        per_year,
    )


history = []
best_selection_score = -1.0
best_epoch = 0
epochs_without_improvement = 0
start_epoch = 1

if LAST_CHECKPOINT_PATH.exists():
    state = torch.load(
        LAST_CHECKPOINT_PATH,
        map_location=device,
        weights_only=False,
    )

    if (
        state.get("artifact_kind")
        != "spatiotemporal_bidirectional_all_year_v4_checkpoint"
    ):
        raise AssertionError(
            "Unexpected checkpoint artifact"
        )

    model.load_state_dict(
        state["model_state_dict"]
    )
    optimizer.load_state_dict(
        state["optimizer_state_dict"]
    )
    scheduler.load_state_dict(
        state["scheduler_state_dict"]
    )

    history = state["history"]
    best_selection_score = float(
        state["best_selection_score"]
    )
    best_epoch = int(
        state["best_epoch"]
    )
    epochs_without_improvement = int(
        state[
            "epochs_without_improvement"
        ]
    )
    start_epoch = (
        int(state["epoch"]) + 1
    )

    print(
        "resuming at epoch:",
        start_epoch,
    )

for epoch in range(
    start_epoch,
    MAX_EPOCHS + 1,
):
    generator = (
        torch.Generator()
        .manual_seed(
            TRAINING_SEED + epoch
        )
    )
    train_loader = DataLoader(
        train_dataset,
        batch_size=FILE_BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=False,
        generator=generator,
    )

    model.train()
    train_loss_sum = 0.0
    train_count = 0
    started = time.perf_counter()

    for batch in train_loader:
        (
            values,
            finite_mask,
            range_folded_mask,
            coordinates,
            targets,
        ) = move_batch_to_device(
            batch
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        with torch.autocast(
            device_type="cuda",
            dtype=torch.bfloat16,
        ):
            logits = model(
                values,
                finite_mask,
                range_folded_mask,
                coordinates,
            )
            loss = criterion(
                logits,
                targets,
            )

        if not torch.isfinite(loss):
            raise RuntimeError(
                f"Non-finite loss at epoch {epoch}"
            )

        loss.backward()
        nn.utils.clip_grad_norm_(
            model.parameters(),
            5.0,
        )
        optimizer.step()

        size = int(
            targets.numel()
        )
        train_loss_sum += (
            float(loss.detach().cpu())
            * size
        )
        train_count += size

    (
        validation_loss,
        validation_pr_auc,
        validation_roc_auc,
        validation_labels,
        validation_probabilities,
    ) = evaluate(
        validation_loader
    )

    (
        validation_pr_auc,
        validation_macro_year_pr_auc,
        validation_selection_score,
        validation_year_pr_auc,
    ) = validation_scores(
        validation_labels,
        validation_probabilities,
    )

    scheduler.step(
        validation_selection_score
    )

    elapsed = (
        time.perf_counter()
        - started
    )
    improved = (
        validation_selection_score
        > best_selection_score
    )

    if improved:
        best_selection_score = (
            validation_selection_score
        )
        best_epoch = epoch
        epochs_without_improvement = 0

        torch.save(
            {
                "artifact_kind": (
                    "spatiotemporal_bidirectional_all_year_v4_best_model"
                ),
                "epoch": epoch,
                "model_state_dict": (
                    model.state_dict()
                ),
                "validation_pr_auc": (
                    validation_pr_auc
                ),
                "validation_macro_year_pr_auc": (
                    validation_macro_year_pr_auc
                ),
                "validation_selection_score": (
                    validation_selection_score
                ),
                "validation_year_pr_auc": (
                    validation_year_pr_auc
                ),
                "pooling_topk_fraction": (
                    POOLING_TOPK_FRACTION
                ),
            },
            BEST_CHECKPOINT_PATH,
        )
    else:
        epochs_without_improvement += 1

    row = {
        "epoch": epoch,
        "train_loss": (
            train_loss_sum
            / train_count
        ),
        "validation_loss": (
            validation_loss
        ),
        "validation_pr_auc": (
            validation_pr_auc
        ),
        "validation_roc_auc": (
            validation_roc_auc
        ),
        "validation_macro_year_pr_auc": (
            validation_macro_year_pr_auc
        ),
        "validation_selection_score": (
            validation_selection_score
        ),
        **{
            (
                f"validation_pr_auc_"
                f"{year}"
            ): validation_year_pr_auc[
                year
            ]
            for year in YEARS
        },
        "learning_rate": (
            optimizer.param_groups[0][
                "lr"
            ]
        ),
        "epoch_seconds": elapsed,
        "improved": improved,
    }
    history.append(row)

    pd.DataFrame(
        history
    ).to_csv(
        HISTORY_PATH,
        index=False,
    )

    torch.save(
        {
            "artifact_kind": (
                "spatiotemporal_bidirectional_all_year_v4_checkpoint"
            ),
            "epoch": epoch,
            "model_state_dict": (
                model.state_dict()
            ),
            "optimizer_state_dict": (
                optimizer.state_dict()
            ),
            "scheduler_state_dict": (
                scheduler.state_dict()
            ),
            "history": history,
            "best_selection_score": (
                best_selection_score
            ),
            "best_epoch": best_epoch,
            "epochs_without_improvement": (
                epochs_without_improvement
            ),
        },
        LAST_CHECKPOINT_PATH,
    )

    print(
        f"epoch={epoch:02d} "
        f"train_loss="
        f"{row['train_loss']:.6f} "
        f"val_loss="
        f"{validation_loss:.6f} "
        f"val_pr_auc="
        f"{validation_pr_auc:.6f} "
        f"val_roc_auc="
        f"{validation_roc_auc:.6f} "
        f"macro_year_pr_auc="
        f"{validation_macro_year_pr_auc:.6f} "
        f"selection_score="
        f"{validation_selection_score:.6f} "
        f"lr="
        f"{row['learning_rate']:.6g} "
        f"seconds={elapsed:.1f} "
        f"best={best_epoch}"
    )

    if (
        epochs_without_improvement
        >= EARLY_STOPPING_PATIENCE
    ):
        print("early stopping")
        break


resuming at epoch: 20
epoch=20 train_loss=0.744897 val_loss=0.809283 val_pr_auc=0.604722 val_roc_auc=0.945471 macro_year_pr_auc=0.606284 selection_score=0.605191 lr=0.0005 seconds=1943.8 best=19
epoch=21 train_loss=0.721767 val_loss=0.765120 val_pr_auc=0.608496 val_roc_auc=0.945400 macro_year_pr_auc=0.609286 selection_score=0.608733 lr=0.0005 seconds=1906.7 best=19
epoch=22 train_loss=0.720847 val_loss=0.793013 val_pr_auc=0.595429 val_roc_auc=0.944860 macro_year_pr_auc=0.597722 selection_score=0.596117 lr=0.00025 seconds=1920.0 best=19
epoch=23 train_loss=0.664715 val_loss=0.778643 val_pr_auc=0.610311 val_roc_auc=0.946405 macro_year_pr_auc=0.612281 selection_score=0.610902 lr=0.00025 seconds=1902.5 best=23
epoch=24 train_loss=0.659447 val_loss=0.883106 val_pr_auc=0.606868 val_roc_auc=0.944896 macro_year_pr_auc=0.608940 selection_score=0.607489 lr=0.00025 seconds=1904.2 best=23
epoch=25 train_loss=0.643316 val_loss=0.913732 val_pr_auc=0.593771 val_roc_auc=0.943121 macro_year_pr_auc=0.59

In [8]:
best_state = torch.load(
    BEST_CHECKPOINT_PATH,
    map_location=device,
    weights_only=False,
)
model.load_state_dict(
    best_state["model_state_dict"]
)

(
    validation_loss,
    validation_pr_auc,
    validation_roc_auc,
    labels,
    probabilities,
) = evaluate(validation_loader)

(
    precision_values,
    recall_values,
    thresholds,
) = precision_recall_curve(
    labels,
    probabilities,
)

f1_values = (
    2
    * precision_values[:-1]
    * recall_values[:-1]
    / np.maximum(
        (
            precision_values[:-1]
            + recall_values[:-1]
        ),
        1e-12,
    )
)
threshold_index = int(
    np.argmax(f1_values)
)
threshold = float(
    thresholds[threshold_index]
)
predictions = (
    probabilities >= threshold
).astype(np.int64)

precision = float(
    precision_score(
        labels,
        predictions,
        zero_division=0,
    )
)
recall = float(
    recall_score(
        labels,
        predictions,
        zero_division=0,
    )
)
f1 = float(
    2
    * precision
    * recall
    / max(
        precision + recall,
        1e-12,
    )
)

matrix = confusion_matrix(
    labels,
    predictions,
    labels=[0, 1],
)
(
    true_negative,
    false_positive,
    false_negative,
    true_positive,
) = [
    int(value)
    for value in matrix.ravel()
]

metrics = {
    "artifact_kind": (
        "spatiotemporal_bidirectional_all_year_v4_"
        "validation_metrics"
    ),
    "created_at_utc": (
        datetime.datetime.now(
            datetime.timezone.utc
        ).isoformat()
    ),
    "package_version": (
        PACKAGE_VERSION
    ),
    "variables": list(VARIABLES),
    "channel_order": list(
        CHANNEL_ORDER
    ),
    "sequence_shape": [
        4,
        2,
        6,
        120,
        240,
    ],
    "coordinate_names": [
        "range_100km",
        "inverse_range_100km",
        "sin_azimuth",
        "cos_azimuth",
        "elevation_degrees",
    ],
    "model_class": (
        "SpatiotemporalTornadoDetector"
    ),
    "file_batch_size": FILE_BATCH_SIZE,
    "pooling_topk_fraction": (
        POOLING_TOPK_FRACTION
    ),
    "bidirectional_temporal": True,
    "temporal_context": (
        "full four-frame offline sequence"
    ),
    "azimuth_roll_augmentation": True,
    "checkpoint_selection": (
        "0.70 overall + 0.30 macro-year "
        "validation PR-AUC"
    ),
    "bfloat16_autocast": True,
    "official_test_evaluated": False,
    "checkpoint_epoch": int(
        best_state["epoch"]
    ),
    "validation_frame_count": int(
        len(labels)
    ),
    "validation_positive_count": int(
        labels.sum()
    ),
    "validation_pr_auc": (
        validation_pr_auc
    ),
    "validation_roc_auc": (
        validation_roc_auc
    ),
    "validation_macro_year_pr_auc": (
        validation_scores(
            labels,
            probabilities,
        )[1]
    ),
    "validation_pr_auc_by_year": (
        validation_scores(
            labels,
            probabilities,
        )[3]
    ),
    "validation_loss": (
        validation_loss
    ),
    "selected_threshold": threshold,
    "threshold_selection": (
        "maximum validation F1"
    ),
    "validation_precision": (
        precision
    ),
    "validation_recall": recall,
    "validation_f1": f1,
    "true_negative": true_negative,
    "false_positive": false_positive,
    "false_negative": false_negative,
    "true_positive": true_positive,
    "normalization_path": str(
        NORMALIZATION_PATH
    ),
    "history_path": str(
        HISTORY_PATH
    ),
    "staging": staging,
}

VALIDATION_METRICS_PATH.write_text(
    json.dumps(
        metrics,
        indent=2,
        sort_keys=True,
    )
    + "\n"
)

print(
    json.dumps(
        metrics,
        indent=2,
        sort_keys=True,
    )
)
print(
    "wrote:",
    VALIDATION_METRICS_PATH,
)

{
  "artifact_kind": "spatiotemporal_bidirectional_all_year_v4_validation_metrics",
  "azimuth_roll_augmentation": true,
  "bfloat16_autocast": true,
  "bidirectional_temporal": true,
  "channel_order": [
    "DBZ_sweep_0",
    "DBZ_sweep_1",
    "KDP_sweep_0",
    "KDP_sweep_1",
    "RHOHV_sweep_0",
    "RHOHV_sweep_1",
    "VEL_sweep_0",
    "VEL_sweep_1",
    "WIDTH_sweep_0",
    "WIDTH_sweep_1",
    "ZDR_sweep_0",
    "ZDR_sweep_1"
  ],
  "checkpoint_epoch": 23,
  "checkpoint_selection": "0.70 overall + 0.30 macro-year validation PR-AUC",
  "coordinate_names": [
    "range_100km",
    "inverse_range_100km",
    "sin_azimuth",
    "cos_azimuth",
    "elevation_degrees"
  ],
  "created_at_utc": "2026-09-29T08:46:11.695627+00:00",
  "false_negative": 1796,
  "false_positive": 1600,
  "file_batch_size": 8,
  "history_path": "/content/drive/MyDrive/TorNet_Backup/experiments/spatiotemporal_bidirectional_all_year_v4/training_history.csv",
  "model_class": "SpatiotemporalTornadoDetector",


In [9]:
shutil.rmtree(EXTRACTION_ROOT)

assert not EXTRACTION_ROOT.exists()
assert BEST_CHECKPOINT_PATH.is_file()
assert LAST_CHECKPOINT_PATH.is_file()
assert HISTORY_PATH.is_file()
assert VALIDATION_METRICS_PATH.is_file()

print(
    "Removed all Colab-local "
    "training data"
)
print(
    "Preserved experiment:",
    EXPERIMENT_DIRECTORY,
)


Removed all Colab-local training data
Preserved experiment: /content/drive/MyDrive/TorNet_Backup/experiments/spatiotemporal_bidirectional_all_year_v4
